In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder,OneHotEncoder,MinMaxScaler
from sklearn.pipeline import Pipeline

In [2]:
car=pd.read_csv("cars.csv")
car.shape

(8128, 5)

In [3]:
car.head()

,brand,km_driven,fuel,owner,selling_price
0,Maruti,145500,Diesel,First Owner,450000
1,Skoda,120000,Diesel,Second Owner,370000
2,Honda,140000,Petrol,Third Owner,158000
3,Hyundai,127000,Diesel,First Owner,225000
4,Maruti,120000,Petrol,First Owner,130000


In [4]:
car.isnull().mean()*100

brand            0.0
km_driven        0.0
fuel             0.0
owner            0.0
selling_price    0.0
dtype: float64

In [5]:
car.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8128 entries, 0 to 8127
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   brand          8128 non-null   object
 1   km_driven      8128 non-null   int64 
 2   fuel           8128 non-null   object
 3   owner          8128 non-null   object
 4   selling_price  8128 non-null   int64 
dtypes: int64(2), object(3)
memory usage: 317.6+ KB


In [6]:
car[car.duplicated()]

,brand,km_driven,fuel,owner,selling_price
291,Hyundai,35000,Petrol,First Owner,450000
296,Maruti,50000,Petrol,Second Owner,330000
370,Jaguar,9000,Diesel,First Owner,2625000
371,Lexus,20000,Petrol,First Owner,5150000
372,Jaguar,45000,Diesel,First Owner,3200000
...,...,...,...,...,...
8115,Maruti,120000,Petrol,First Owner,40000
8117,Maruti,50000,Diesel,First Owner,625000
8118,Hyundai,25000,Petrol,First Owner,380000
8126,Tata,25000,Diesel,First Owner,290000


In [7]:
car.drop_duplicates(keep='first',inplace=True)

In [8]:
car.shape

(6450, 5)

In [9]:
car.describe().T

,count,mean,std,min,25%,50%,75%,max
km_driven,6450.0,74469.011318,59633.411745,1.0,39000.0,70000.0,100000.0,2360457.0
selling_price,6450.0,524229.724341,534126.687173,29999.0,250000.0,409999.0,640000.0,10000000.0


In [10]:
np.random.seed(42)
missing_km_indices = np.random.choice(car.index, size=int(0.05*len(car)), replace=False)
car.loc[missing_km_indices, 'km_driven'] = np.nan

# Introduce missing values in 'owner' column (1% missing values)
missing_owner_indices = np.random.choice(car.index, size=int(0.01*len(car)), replace=False)
car.loc[missing_owner_indices, 'owner'] = np.nan

In [11]:
car.isnull().mean()*100

brand            0.000000
km_driven        4.992248
fuel             0.000000
owner            0.992248
selling_price    0.000000
dtype: float64

In [12]:
catergorical_col=[cols for cols in car.columns if car[cols].dtype=="object"]
catergorical_col

['brand', 'fuel', 'owner']

In [13]:
for i in car[catergorical_col]:
    print(car[i].unique())
    print("*"*100)

['Maruti' 'Skoda' 'Honda' 'Hyundai' 'Toyota' 'Ford' 'Renault' 'Mahindra'
 'Tata' 'Chevrolet' 'Fiat' 'Datsun' 'Jeep' 'Mercedes-Benz' 'Mitsubishi'
 'Audi' 'Volkswagen' 'BMW' 'Nissan' 'Lexus' 'Jaguar' 'Land' 'MG' 'Volvo'
 'Daewoo' 'Kia' 'Force' 'Ambassador' 'Ashok' 'Isuzu' 'Opel' 'Peugeot']
****************************************************************************************************
['Diesel' 'Petrol' 'LPG' 'CNG']
****************************************************************************************************
['First Owner' 'Second Owner' 'Third Owner' nan 'Fourth & Above Owner'
 'Test Drive Car']
****************************************************************************************************


In [14]:
X=car.drop(columns=['selling_price'])
Y=car['selling_price']

In [15]:
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,random_state=10,test_size=0.2)

In [16]:
missing_imputation=ColumnTransformer(
    [("imputation_num",SimpleImputer(),[1]),
    ("imputation_cat",SimpleImputer(strategy="most_frequent"),[3])],remainder="passthrough"
)

In [17]:
encoder=ColumnTransformer(
    [("Ordinal",OrdinalEncoder(handle_unknown='use_encoded_value',unknown_value=-1),[3]),
    ("OHE",OneHotEncoder(sparse_output=False,handle_unknown='ignore',drop="first"),[0,2])],remainder="passthrough"
)

In [18]:
scaler=ColumnTransformer(
    [("minmax",MinMaxScaler(),slice(0,37))]
)

In [19]:
lr=LinearRegression()

In [20]:
pipeline1=Pipeline([
    ("imputation",missing_imputation),
    ("encode",encoder),
    ("scale",scaler),
    ("model",lr)]
)

In [21]:
pipeline1.fit(X_train,Y_train)

Pipeline(steps=[('imputation',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('imputation_num',
                                                  SimpleImputer(), [1]),
                                                 ('imputation_cat',
                                                  SimpleImputer(strategy='most_frequent'),
                                                  [3])])),
                ('encode',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('Ordinal',
                                                  OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                 unknown_value=-1),
                                                  [3]),
                                                 ('OHE',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore',
                                                                sparse_output=False),
                                                  [0, 2])])),
                ('scale',
                 ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                                  slice(0, 37, None))])),
                ('model', LinearRegression())])

In [22]:
pipeline1.feature_names_in_

array(['brand', 'km_driven', 'fuel', 'owner'], dtype=object)

In [23]:
pipeline1.predict(X_test)

C:\Users\khadk\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


array([634865.55505886, 634865.55505886, 370376.77058593, ...,
       370376.77058593, 634865.55505886, 634865.55505886], shape=(1290,))

In [24]:
pipeline1.predict(np.array(['Hyundai',35000,'Petrol','First Owner']).reshape(1,4))

C:\Users\khadk\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but SimpleImputer was fitted with feature names
  warnings.warn(
C:\Users\khadk\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but SimpleImputer was fitted with feature names
  warnings.warn(


array([370376.77058593])